In [ ]:
'\u4e00-\u9fff'

In [ ]:
dict.setdefault

In [ ]:
from json import loads
import numpy as np


def f():
    global m
    m = {"。": 0}
    for i in "train", "valid":
        for i in open(f"../tmp/news2016zh_{i}.json"):
            i = loads(i)
            for i in i["keywords"], i["desc"], i["title"], i["source"], i["content"]:
                for i in i:
                    yield m.setdefault(i, len(m))
                if '\u4e00' <= i <= '\u9fff':
                    yield 0


a = np.fromiter(f(), "H")

In [ ]:
open(f"../tmp/news2016zh_{i}.H", "wb").writelines(i for i in ['train','valid']for i in open(f'../tmp/news2016zh_{i}.json')for i in [loads(i)]for i in [i['keywords'],i['desc'],i['title'],i['source'],i['content']]

In [ ]:
from re import compile
from json import loads
m = {}
r = compile("[\u4e00-\u9fa5]")
for i in "train", "valid":
    
    open(f"../tmp/news2016zh_{i}.H", "wb").writelines(
        m.setdefault(i, len(m)).to_bytes(2, "little") for i in open(f"../tmp/news2016zh_{i}.json") for i in r.findall(loads(i)["content"])
    )
len(m)

In [ ]:
pd.read_json('../tmp/news2016zh_train.json', lines=1, nrows=20)

```python
from re import compile
from json import loads
m = {}
r = compile("[\u4e00-\u9fa5]")
for i in "train", "valid":
    open(f"../tmp/news2016zh_{i}.H", "wb").writelines(
        m.setdefault(i, len(m)).to_bytes(2, "little") for i in open(f"../tmp/news2016zh_{i}.json") for i in r.findall(loads(i)["content"])
    )
len(m)
```

In [ ]:
from fastai.vision.all import *
from torch.nn import *


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zh.H", "h"))

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w, w + 1], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x[:, :-1], x[:, -1]


class F(Module):
    def __init__(s, f):
        super().__init__()
        s.forward = f
        for i, f in zip(f.__code__.co_varnames[1:], f.__defaults__ or ()):
            setattr(s, i, f)

In [ ]:
m = Learner(
    DataLoaders(Dat(8, 2048 * 8, 10), Dat(8, 2048 * 8, 1000)),
    Sequential(
        Embedding(21482, 32),
        *[F(lambda x: x.view(-1, 64)), Linear(64, 32, 0), BatchNorm1d(32, track_running_stats=0), Swish()] * 3,
        Linear(32, 32, 0),
        BatchNorm1d(32, track_running_stats=0),
        Swish(),
        Linear(32, 21482)
    ).cuda(),
    CrossEntropyLoss(),
    metrics=accuracy,
    cbs=[ShowGraphCallback, SaveModelCallback(fname="model2")],
)
m.lr_find()

In [ ]:
m.fit_one_cycle(100, .01, 9, 999, .01)